<a href="https://colab.research.google.com/github/h1baq1m/skills-getting-started-with-github-copilot/blob/main/AssociationRule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#install apyori and import it
!pip install apyori
from apyori import apriori
import pandas as pd

In [6]:
# read bakery dataset and find out if data is suitable for association rule
df = pd.read_csv("Bakery.csv")

print("Dimensions of dataset are:", df.shape)
print(df.head())
print("Number of transactions:", df["TransactionNo"].nunique())
print("Number of unique bakery items:", df["Items"].nunique())

Dimensions of dataset are: (20507, 5)
   TransactionNo          Items             DateTime  Daypart  DayType
0              1          Bread  2016-10-30 09:58:11  Morning  Weekend
1              2   Scandinavian  2016-10-30 10:05:34  Morning  Weekend
2              2   Scandinavian  2016-10-30 10:05:34  Morning  Weekend
3              3  Hot chocolate  2016-10-30 10:07:57  Morning  Weekend
4              3            Jam  2016-10-30 10:07:57  Morning  Weekend
Number of transactions: 9465
Number of unique bakery items: 94


In [7]:
#find out if values are good enough to make rules
pd.set_option("display.max_rows", None)
print(df["Items"].value_counts())

Items
Coffee                           5471
Bread                            3325
Tea                              1435
Cake                             1025
Pastry                            856
Sandwich                          771
Medialuna                         616
Hot chocolate                     590
Cookies                           540
Brownie                           379
Farm House                        374
Muffin                            370
Alfajores                         369
Juice                             369
Soup                              342
Scone                             327
Toast                             318
Scandinavian                      277
Truffles                          193
Coke                              185
Spanish Brunch                    172
Fudge                             159
Baguette                          152
Jam                               149
Tiffin                            146
Mineral water                     136
Jammie

In [8]:
# Task 3 - Check for missing data and any duplicated records.
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

# Check for missing data.
missing_data = df.isnull().sum()
print("Missing data in each column:")
print(missing_data)

Number of duplicate rows: 1620
Missing data in each column:
TransactionNo    0
Items            0
DateTime         0
Daypart          0
DayType          0
dtype: int64


In [9]:
# group the data by transaction no
df["DateTime"] = pd.to_datetime(df["DateTime"])

def unique_items(items):
    return list(set(items))

All_Transactions_Group = df.groupby("TransactionNo")["Items"].apply(unique_items).reset_index()
Weekday_Group = df[df["DayType"] == "Weekday"].groupby("TransactionNo")["Items"].apply(unique_items).reset_index()
Weekend_Group = df[df["DayType"] == "Weekend"].groupby("TransactionNo")["Items"].apply(unique_items).reset_index()

print("All transactions:")
print(All_Transactions_Group.head())

print("Weekday transactions:")
print(Weekday_Group.head())

print("Weekend transactions:")
print(Weekend_Group.head())

All transactions:
   TransactionNo                          Items
0              1                        [Bread]
1              2                 [Scandinavian]
2              3  [Cookies, Hot chocolate, Jam]
3              4                       [Muffin]
4              5        [Pastry, Coffee, Bread]
Weekday transactions:
   TransactionNo                        Items
0             81               [Cake, Coffee]
1             82             [Tartine, Bread]
2             83              [Coffee, Bread]
3             84                      [Bread]
4             85  [Pastry, Coffee, Medialuna]
Weekend transactions:
   TransactionNo                          Items
0              1                        [Bread]
1              2                 [Scandinavian]
2              3  [Cookies, Hot chocolate, Jam]
3              4                       [Muffin]
4              5        [Pastry, Coffee, Bread]


In [10]:
# Convert the grouped transaction baskets into lists for the Apriori algorithm.
All_transactions = All_Transactions_Group["Items"].to_list()
Weekday_transactions = Weekday_Group["Items"].to_list()
Weekend_transactions = Weekend_Group["Items"].to_list()

# Create Apriori rules.
All_rules = apriori(
    transactions=All_transactions,
    min_support=0.01,
    min_confidence=0.2,
    min_lift=1.0,
    min_length=2,
    max_length=2
)

Weekday_rules = apriori(
    transactions=Weekday_transactions,
    min_support=0.01,
    min_confidence=0.2,
    min_lift=1.0,
    min_length=2,
    max_length=2
)

Weekend_rules = apriori(
    transactions=Weekend_transactions,
    min_support=0.01,
    min_confidence=0.2,
    min_lift=1.0,
    min_length=2,
    max_length=2
)

In [11]:
# Displaying the first results coming directly from the output of the apriori function.
All_results = list(All_rules)
Weekday_results = list(Weekday_rules)
Weekend_results = list(Weekend_rules)

print(All_results[:5])

# Convert the Apriori results into a Pandas DataFrame.
def inspect(results):
    rows = []
    for result in results:
        support = result.support
        for ordered_stat in result.ordered_statistics:
            lhs = tuple(ordered_stat.items_base)
            rhs = tuple(ordered_stat.items_add)
            if len(lhs) > 0 and len(rhs) > 0:
                rows.append([
                    ", ".join(lhs),
                    ", ".join(rhs),
                    support,
                    ordered_stat.confidence,
                    ordered_stat.lift
                ])
    return rows

All_results_DataFrame = pd.DataFrame(
    inspect(All_results),
    columns=["Left Hand Side", "Right Hand Side", "Support", "Confidence", "Lift"]
)

Weekday_results_DataFrame = pd.DataFrame(
    inspect(Weekday_results),
    columns=["Left Hand Side", "Right Hand Side", "Support", "Confidence", "Lift"]
)

Weekend_results_DataFrame = pd.DataFrame(
    inspect(Weekend_results),
    columns=["Left Hand Side", "Right Hand Side", "Support", "Confidence", "Lift"]
)

[RelationRecord(items=frozenset({'Bread'}), support=0.32720549392498677, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Bread'}), confidence=0.32720549392498677, lift=1.0)]), RelationRecord(items=frozenset({'Coffee'}), support=0.47839408346539886, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Coffee'}), confidence=0.47839408346539886, lift=1.0)]), RelationRecord(items=frozenset({'Alfajores', 'Coffee'}), support=0.0196513470681458, ordered_statistics=[OrderedStatistic(items_base=frozenset({'Alfajores'}), items_add=frozenset({'Coffee'}), confidence=0.5406976744186046, lift=1.1302348693401265)]), RelationRecord(items=frozenset({'Pastry', 'Bread'}), support=0.029160063391442156, ordered_statistics=[OrderedStatistic(items_base=frozenset({'Pastry'}), items_add=frozenset({'Bread'}), confidence=0.33865030674846625, lift=1.0349774470049187)]), RelationRecord(items=frozenset({'Brownie', 'Coffee'}), support=0.019651347068145

In [12]:
# Displaying the results for all bakery transactions.
print("Rules for all bakery transactions")
print(All_results_DataFrame.sort_values("Confidence", ascending=False))

Rules for all bakery transactions
    Left Hand Side Right Hand Side   Support  Confidence      Lift
14           Toast          Coffee  0.023666    0.704403  1.472431
13  Spanish Brunch          Coffee  0.010882    0.598837  1.251766
8        Medialuna          Coffee  0.035182    0.569231  1.189878
10          Pastry          Coffee  0.047544    0.552147  1.154168
0        Alfajores          Coffee  0.019651    0.540698  1.130235
7            Juice          Coffee  0.020602    0.534247  1.116750
11        Sandwich          Coffee  0.038246    0.532353  1.112792
3             Cake          Coffee  0.054728    0.526958  1.101515
12           Scone          Coffee  0.018067    0.522936  1.093107
5          Cookies          Coffee  0.028209    0.518447  1.083723
6    Hot chocolate          Coffee  0.029583    0.507246  1.060311
2          Brownie          Coffee  0.019651    0.490765  1.025860
9           Muffin          Coffee  0.018806    0.489011  1.022193
1           Pastry          

In [13]:
# Displaying the results for weekday bakery transactions.
print("Rules for weekday bakery transactions")
print(Weekday_results_DataFrame.sort_values("Confidence", ascending=False))

Rules for weekday bakery transactions
   Left Hand Side Right Hand Side   Support  Confidence      Lift
11          Toast          Coffee  0.026526    0.711790  1.465802
8          Pastry          Coffee  0.049634    0.566914  1.167456
0       Alfajores          Coffee  0.019365    0.550926  1.134531
1            Cake          Coffee  0.052238    0.544992  1.122310
6       Medialuna          Coffee  0.028641    0.543210  1.118641
10          Scone          Coffee  0.010578    0.524194  1.079480
7          Muffin          Coffee  0.017738    0.511737  1.053829
5           Juice          Coffee  0.018714    0.506608  1.043266
3         Cookies          Coffee  0.031082    0.501312  1.032361
9        Sandwich          Coffee  0.037592    0.498920  1.027434
4   Hot chocolate          Coffee  0.025386    0.493671  1.016625
14           Soup             Tea  0.011229    0.269531  1.804215
2            Cake             Tea  0.022783    0.237691  1.591080
13       Sandwich             Tea  0.0

In [14]:
# Displaying the results for weekend bakery transactions.
print("Rules for weekend bakery transactions")
print(Weekend_results_DataFrame.sort_values("Confidence", ascending=False))

Rules for weekend bakery transactions
    Left Hand Side Right Hand Side   Support  Confidence      Lift
14  Spanish Brunch          Coffee  0.021386    0.702970  1.511568
16           Toast          Coffee  0.018373    0.685393  1.473773
13            Soup          Coffee  0.012952    0.614286  1.320873
11        Sandwich          Coffee  0.039458    0.603687  1.298083
9        Medialuna          Coffee  0.047289    0.601533  1.293451
15          Tiffin          Coffee  0.011145    0.587302  1.262851
8            Juice          Coffee  0.024096    0.579710  1.246527
6          Cookies          Coffee  0.022892    0.567164  1.219550
2          Brownie          Coffee  0.028614    0.552326  1.187643
7    Hot chocolate          Coffee  0.037349    0.525424  1.129797
10          Pastry          Coffee  0.043675    0.523466  1.125587
0        Alfajores          Coffee  0.020181    0.523437  1.125526
12           Scone          Coffee  0.031928    0.522167  1.122795
3             Cake      